<a href="https://colab.research.google.com/github/heyanugrah/Transformers/blob/main/TransformerTraining.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

Transformer - Translation






In [1]:
import torch
import torch.nn as nn
import math

In [2]:

device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print(f"Using device: {device}")

Using device: cuda


In [3]:
# Helper to add special tokens and initial segment IDs
def _add_special_tokens_and_segments(input_text, vocab):
    tokens = [vocab.get("<cls>", "<cls>")] # Start with <cls>
    segment_ids = [0] # <cls> belongs to segment 0

    # Tokenize input text and add words
    for word in input_text.lower().replace('.', '').split():
        tokens.append(word)
        segment_ids.append(0) # All words in the first sentence belong to segment 0

    tokens.append(vocab.get("[sep]", "[sep]")) # Add [sep]
    segment_ids.append(0) # [sep] belongs to segment 0

    return tokens, segment_ids

# Helper to convert tokens to IDs
def _convert_to_ids(tokens, vocab):
    return [vocab.get(token, vocab["<unk>"]) for token in tokens]

# Helper to pad token IDs and segment IDs
def _pad_ids_and_segments(token_ids, segment_ids, max_seq_len, pad_token_id):
    if len(token_ids) > max_seq_len:
        token_ids = token_ids[:max_seq_len]
        segment_ids = segment_ids[:max_seq_len]
    else:
        padding_length = max_seq_len - len(token_ids)
        token_ids.extend([pad_token_id] * padding_length)
        segment_ids.extend([0] * padding_length) # Padding tokens belong to segment 0

    return token_ids, segment_ids

# Helper to create attention mask
def _create_attention_mask_from_ids(token_ids, pad_token_id):
    mask = [1 if token_id != pad_token_id else 0 for token_id in token_ids]
    return torch.tensor(mask, dtype=torch.long).unsqueeze(0).unsqueeze(1).unsqueeze(2) # [batch_size, 1, 1, seq_len]

In [4]:
def prepare_transformer_inputs(input_text, vocab, max_seq_len):
    pad_token_id = vocab['<pad>']

    # Step 1: Add special tokens and assign initial segment IDs
    tokens, segment_ids = _add_special_tokens_and_segments(input_text, vocab)

    # Step 2: Convert tokens to IDs
    token_ids = _convert_to_ids(tokens, vocab)

    # Step 3: Pad token IDs and segment IDs
    padded_token_ids, padded_segment_ids = _pad_ids_and_segments(
        token_ids, segment_ids, max_seq_len, pad_token_id
    )

    # Convert to PyTorch tensors and add batch dimension
    input_ids = torch.tensor(padded_token_ids, dtype=torch.long).unsqueeze(0)
    token_type_ids = torch.tensor(padded_segment_ids, dtype=torch.long).unsqueeze(0)

    # Step 4: Create attention mask
    attention_mask = _create_attention_mask_from_ids(padded_token_ids, pad_token_id)

    return input_ids, attention_mask, token_type_ids

In [5]:
class PositionalEncoding(nn.Module):
    def __init__(self, d_model, max_seq_len=5000):
        super().__init__()

        self.dropout = nn.Dropout(0.1)

        # Create a positional encoding matrix
        pe = torch.zeros(max_seq_len, d_model)
        position = torch.arange(0, max_seq_len, dtype=torch.float).unsqueeze(1)
        div_term = torch.exp(torch.arange(0, d_model, 2).float() * (-math.log(10000.0) / d_model))

        pe[:, 0::2] = torch.sin(position * div_term)
        pe[:, 1::2] = torch.cos(position * div_term) if d_model % 2 == 0 else torch.cos(position * div_term[:-1])

        pe = pe.unsqueeze(0) # Add batch dimension
        self.register_buffer('pe', pe)

    def forward(self, x):
        # Add positional encoding to the input embeddings
        # x has shape (batch_size, seq_len, d_model)
        x = x + self.pe[:, :x.size(1)]
        return self.dropout(x)


In [6]:
class TransformerEncoderLayer(nn.Module):
    def __init__(self, d_model, num_heads, ffn_hidden_dim, dropout_rate=0.1):
        super().__init__()

        '''
          d_model = 512
          num_heads = 8

          head_dim = 64
          512 dimensions
          ├─ Head 1 : 64
          ├─ Head 2 : 64
          ├─ Head 3 : 64
          ├─ Head 4 : 64
          ├─ Head 5 : 64
          ├─ Head 6 : 64
          ├─ Head 7 : 64
          └─ Head 8 : 64
        '''

        assert d_model % num_heads == 0 #if it is divisible then return true and allow run

        self.d_model = d_model
        self.num_heads = num_heads
        self.head_dim = d_model // num_heads

        # Multi-Head Attention
        self.Wq = nn.Linear(d_model, d_model)
        self.Wk = nn.Linear(d_model, d_model)
        self.Wv = nn.Linear(d_model, d_model)
        self.Wo = nn.Linear(d_model, d_model)

        self.attn_dropout = nn.Dropout(dropout_rate)

        # LayerNorm after MHA residual
        self.layer_norm1 = nn.LayerNorm(d_model)

        # Feed Forward Network
        self.ffn = nn.Sequential(
            nn.Linear(d_model, ffn_hidden_dim),
            nn.ReLU(),
            nn.Linear(ffn_hidden_dim, d_model)
        )

        self.ffn_dropout = nn.Dropout(dropout_rate)

        # LayerNorm after FFN residual
        self.layer_norm2 = nn.LayerNorm(d_model)

    def _calculate_attention(self, Q, K, V, mask=None):

        scores = torch.matmul(
            Q,
            K.transpose(-2, -1)
        ) / math.sqrt(self.head_dim) # The division by sqrt(head_dim) is the scaling factor.

        if mask is not None:
            # Apply padding mask to the scores. Positions where mask == 0 are set to -inf.
            # This prevents attention to padded tokens in the input sequence.
            scores = scores.masked_fill(mask == 0, float('-inf'))

        attention_weights = torch.softmax(
            scores,
            dim=-1
        )

        output = torch.matmul(
            attention_weights,
            V
        )

        return output

    def forward(self, x, mask=None):
        # In your current setup (cell cbe1d5c1), `mask` is indeed passed as a tensor
        # (attention_mask_new), not None, when calling this layer.

        batch_size, seq_len, _ = x.shape

        # ----------------------------
        # Multi-Head Self Attention
        # ----------------------------

        Q = self.Wq(x)
        K = self.Wk(x)
        V = self.Wv(x)

        '''
        Full token embedding (8 dims)
                  │
                  ▼
            Split into 2 heads
                  │
          ┌──────┴──────┐
          ▼             ▼
        Head 1        Head 2
        (4 dims)      (4 dims)
          │             │
        Attention    Attention
        separately   separately
        '''

        '''
        # Split Q into multiple attention heads and rearrange dimensions
        # From: [batch_size, seq_len, d_model]
        # To:   [batch_size, num_heads, seq_len, head_dim]
        '''
        Q = Q.view(
            batch_size,
            seq_len,
            self.num_heads,
            self.head_dim
        ).transpose(1, 2)

        K = K.view(
            batch_size,
            seq_len,
            self.num_heads,
            self.head_dim
        ).transpose(1, 2)

        V = V.view(
            batch_size,
            seq_len,
            self.num_heads,
            self.head_dim
        ).transpose(1, 2)

        attention_output = self._calculate_attention(
            Q,
            K,
            V,
            mask=mask
        )
        '''
        Before attention:

        Token
        [1,2,3,4,5,6,7,8]

        ↓ split

        Head1 [1,2,3,4]
        Head2 [5,6,7,8]

        After attention:
        Head1 [1,2,3,4]
        Head2 [5,6,7,8]

        ↓ concatenate
        combined attention heads : [1,2,3,4,5,6,7,8]
        '''
        attention_output = (
            attention_output
            .transpose(1, 2)
            .contiguous()
            .view(
                batch_size,
                seq_len,
                self.d_model
            )
        )
        '''
        Multi-head outputs
              ↓
        Concatenate
              ↓
        Wo (Linear Layer)
              ↓
        Final attention representation

        [1,2,3,4,5,6,7,8]
            ↓
        Linear Wo
            ↓
       [5,-2,8,1,4,9,3,7]
        '''

        mha_output = self.Wo(
            attention_output
        )
        '''
        Some output values are randomly set to 0.Prevent Overfitting

        Before:
        [5,-2,8,1,4,9,3,7]

        After dropout:
        [5,0,8,1,0,9,3,7]
        '''
        final_mha_output = self.attn_dropout(
            mha_output
        )

        '''
          Input
            ↓
          Wq, Wk, Wv
            ↓
          Split into heads
            ↓
          Attention
            ↓
          Merge heads
            ↓
          Wo
            ↓
          Dropout
            ↓
          Final MHA output

        '''

        # Residual Connection + LayerNorm
        x = self.layer_norm1(
            x + final_mha_output
        )

        # ----------------------------
        # Feed Forward Network
        # ----------------------------

        ffn_output = self.ffn(x)

        ffn_output = self.ffn_dropout(
            ffn_output
        )

        # Residual Connection + LayerNorm
        output = self.layer_norm2(
            x + ffn_output
        )

        return output

# # --- Execution Code ---
# encoder_layer = TransformerEncoderLayer(d_model=8, num_heads=2, ffn_hidden_dim=32)
# encoder_output = encoder_layer(final_transformer_input, mask=attention_mask_new)

# print(f"Encoder Layer Input Shape: {final_transformer_input.shape}")
# print(f"Encoder Layer Output Shape: {encoder_output.shape}")
# print("\nFirst few values of the output:")
# print(encoder_output[0, 0, :])

In [7]:
'''
Input Target Sentence
        ↓
Embedding
        ↓
Positional Encoding
        ↓
Masked Self Attention
(Q,K,V from Decoder Input)
        ↓
Add + Norm
        ↓
Cross Attention
(Q from Decoder)

(K,V from Encoder Output)
        ↓
Add + Norm
        ↓
Feed Forward Network
        ↓
Add + Norm
        ↓
Decoder Output
'''

class TransformerDecoderLayer(nn.Module):

    def __init__(self,
                 d_model,
                 num_heads,
                 ffn_hidden_dim,
                 dropout_rate=0.1):

        super().__init__()

        self.d_model = d_model
        self.num_heads = num_heads
        self.head_dim = d_model // num_heads

        # ===============================
        # 1. Masked Self Attention
        # ===============================

        self.self_Wq = nn.Linear(d_model, d_model)
        self.self_Wk = nn.Linear(d_model, d_model)
        self.self_Wv = nn.Linear(d_model, d_model)

        self.self_Wo = nn.Linear(d_model, d_model)

        self.norm1 = nn.LayerNorm(d_model)
        self.dropout1 = nn.Dropout(dropout_rate)

        # ===============================
        # 2. Encoder Decoder Attention
        # ===============================

        self.cross_Wq = nn.Linear(d_model, d_model)

        self.cross_Wk = nn.Linear(d_model, d_model)
        self.cross_Wv = nn.Linear(d_model, d_model)

        self.cross_Wo = nn.Linear(d_model, d_model)

        self.norm2 = nn.LayerNorm(d_model)
        self.dropout2 = nn.Dropout(dropout_rate)

        # ===============================
        # 3. Feed Forward Network
        # ===============================

        self.ffn = nn.Sequential(
            nn.Linear(d_model, ffn_hidden_dim),
            nn.ReLU(),
            nn.Linear(ffn_hidden_dim, d_model)
        )

        self.norm3 = nn.LayerNorm(d_model)
        self.dropout3 = nn.Dropout(dropout_rate)

    """
    Splits the embedding dimension (d_model) into multiple attention heads.

    Input Shape:
        (batch_size, seq_len, d_model)

    Example:
        (32, 10, 512)

    After Reshape:
        (batch_size, seq_len, num_heads, head_dim)

    Example:
        (32, 10, 8, 64)

    After Transpose:
        (batch_size, num_heads, seq_len, head_dim)

    Example:
        (32, 8, 10, 64)

    Purpose:
        Allows Multi-Head Attention to process multiple attention heads
        in parallel, where each head learns different relationships
        between tokens.
    """
    def split_heads(self, x):


        batch_size = x.shape[0]
        seq_len = x.shape[1]

        x = x.view(
            batch_size,
            seq_len,
            self.num_heads,
            self.head_dim
        )

        return x.transpose(1, 2)

    """
    Computes Scaled Dot-Product Attention.

    Steps:
    1. Calculate attention scores between Query (Q) and Key (K).
       Higher scores indicate stronger relationships between tokens.

    2. Scale the scores by sqrt(head_dim) to prevent very large values
       that can make softmax unstable.

    3. Apply the mask (if provided):
       - src_mask: ignores padding tokens.
       - tgt_mask: ignores padding tokens and future tokens.

    4. Apply Softmax to convert scores into attention weights
       (probabilities that sum to 1).

    5. Multiply attention weights by Value (V) to produce the
       final attention output.

    Input Shapes:
        Q : (batch_size, num_heads, seq_len_q, head_dim)
        K : (batch_size, num_heads, seq_len_k, head_dim)
        V : (batch_size, num_heads, seq_len_k, head_dim)

    Output Shape:
        (batch_size, num_heads, seq_len_q, head_dim)

    Formula:
        Attention(Q,K,V) =
        softmax((Q × Kᵀ) / √head_dim) × V

    Args:
        Q    : Query vectors.
        K    : Key vectors.
        V    : Value vectors.
        mask : Optional attention mask. If None, attention is
               computed normally without masking.
    """
    def attention(self, Q, K, V, mask=None):

        scores = torch.matmul(
            Q,
            K.transpose(-2, -1)
        )

        scores = scores / math.sqrt(self.head_dim)

        if mask is not None:
            scores = scores.masked_fill(
                mask == 0,
                float("-inf")
            )

        weights = torch.softmax(scores, dim=-1)

        output = torch.matmul(weights, V)

        return output

    """
    Performs one Transformer Decoder Layer forward pass.

    Args:
        decoder_input:
            Target sequence representation from the previous decoder
            layer (or embedding layer for the first decoder layer).

        encoder_output:
            Output produced by the encoder. Used by the decoder
            to access information from the source sequence.

        src_mask:
            Source padding mask used during encoder-decoder attention.
            Prevents the decoder from attending to PAD tokens in the
            encoder output.

        tgt_mask:
            Target mask used during masked self-attention.
            Combines:
            - Look-ahead (causal) mask to hide future tokens.
            - Padding mask to ignore PAD tokens.

    Processing Steps:

    Step 1: Masked Multi-Head Self-Attention
        - Generate Q, K, and V from decoder_input.
        - Apply tgt_mask to prevent attention to future words.
        - Combine attention heads.
        - Apply output projection (Wo).
        - Apply Residual Connection and LayerNorm.

    Step 2: Encoder-Decoder Multi-Head Attention
        - Q comes from decoder output.
        - K and V come from encoder output.
        - Apply src_mask to ignore encoder PAD tokens.
        - Allows the decoder to attend to relevant source tokens.
        - Apply Residual Connection and LayerNorm.

    Step 3: Feed Forward Network
        - Apply two fully connected layers with activation.
        - Process each token independently.
        - Apply Residual Connection and LayerNorm.

    Returns:
        decoder_output:
            Final decoder representation after masked self-attention,
            encoder-decoder attention, and feed-forward processing.

    Architecture:

        decoder_input
              ↓
      Masked Self-Attention
          (Q,K,V from Decoder)
              ↓
            Add & Norm
              ↓
      Encoder-Decoder Attention
      (Q from Decoder, K/V from Encoder)
              ↓
            Add & Norm
              ↓
       Feed Forward Network
              ↓
            Add & Norm
              ↓
         decoder_output
    """
    def forward(self,
                decoder_input,
                encoder_output,
                src_mask,
                tgt_mask):

        batch_size = decoder_input.shape[0]
        decoder_seq_len = decoder_input.shape[1]

        # ==========================================
        # STEP 1 : Masked Self Attention
        # ==========================================
        '''
            # Generate Query (Q), Key (K), and Value (V) vectors
            # from the decoder input using learned linear projections.
            #
            # Shape:
            # (batch_size, seq_len, d_model)
            #           ↓
            # (batch_size, seq_len, d_model)
            #
            # Then split each projection into multiple attention heads.
            #
            # Example:
            # (32, 10, 512)
            #        ↓
            # (32, 8, 10, 64)
            #
            # This allows each attention head to learn different
            # relationships and patterns between tokens.
        '''
        Q = self.self_Wq(decoder_input)
        K = self.self_Wk(decoder_input)
        V = self.self_Wv(decoder_input)

        Q = self.split_heads(Q)
        K = self.split_heads(K)
        V = self.split_heads(V)

        # Compute masked self-attention.
        # Q, K, and V are generated from the decoder input.
        # tgt_mask prevents attention to future tokens and PAD tokens.
        '''
        Target sequence(tgt_mask):

        <START> I love PAD PAD

        Padding mask:

        1 1 1 0 0

        Look-ahead mask:

        1 0 0 0 0
        1 1 0 0 0
        1 1 1 0 0
        1 1 1 1 0
        1 1 1 1 1

        Combined:

        1 0 0 0 0
        1 1 0 0 0
        1 1 1 0 0
        0 0 0 0 0
        0 0 0 0 0
        '''
        attention_output = self.attention(
            Q,
            K,
            V,
            tgt_mask
        )

        # Combine all attention heads back into a single d_model vector.
        #
        # Shape transformation:
        # (batch_size, num_heads, seq_len, head_dim)
        #                ↓
        # (batch_size, seq_len, num_heads, head_dim)
        #                ↓
        # (batch_size, seq_len, d_model)
        #
        # Example:
        # (32, 8, 10, 64) → (32, 10, 512)

        # Multi-Head Attention produces separate outputs for each head.
        # Transpose moves the num_heads dimension back after seq_len.
        # Then all heads are concatenated together to reconstruct
        # the original d_model representation.
        #
        # Example:
        # Head1(64) + Head2(64) + ... + Head8(64)
        #                ↓
        #            512 features
        attention_output = (
            attention_output
            .transpose(1, 2)
            .contiguous()
            .view(
                batch_size,
                decoder_seq_len,
                self.d_model
            )
        )

        attention_output = self.self_Wo(
            attention_output
        )

        attention_output = self.dropout1(
            attention_output
        )

        decoder_output = self.norm1(
            decoder_input + attention_output
        )

        # ==========================================
        # STEP 2 : Encoder Decoder Attention
        # ==========================================

        encoder_seq_len = encoder_output.shape[1]

        # Generate Query from decoder output
        Q = self.cross_Wq(
            decoder_output
        )

        # Generate Key and Value from encoder output
        K = self.cross_Wk(
            encoder_output
        )

        V = self.cross_Wv(
            encoder_output
        )

        # Split Query into multiple attention heads
        Q = self.split_heads(Q)

        # Split Key into multiple attention heads
        K = K.view(
            batch_size,
            encoder_seq_len,
            self.num_heads,
            self.head_dim
        ).transpose(1, 2)

        # Split Value into multiple attention heads
        V = V.view(
            batch_size,
            encoder_seq_len,
            self.num_heads,
            self.head_dim
        ).transpose(1, 2)

        #Compute cross-attention using decoder queries
        # and encoder keys/values.
        cross_output = self.attention(
            Q,
            K,
            V,
            src_mask
        )

        # Concatenate all attention heads back into d_model
        cross_output = (
            cross_output
            .transpose(1, 2)
            .contiguous()
            .view(
                batch_size,
                decoder_seq_len,
                self.d_model
            )
        )

        # Output projection
        cross_output = self.cross_Wo(
            cross_output
        )

        # Apply dropout
        cross_output = self.dropout2(
            cross_output
        )

        # Residual connection + Layer Normalization
        decoder_output = self.norm2(
            decoder_output + cross_output
        )

        # ==========================================
        # STEP 3 : Feed Forward Network
        # ==========================================

        ffn_output = self.ffn(
            decoder_output
        )

        ffn_output = self.dropout3(
            ffn_output
        )

        decoder_output = self.norm3(
            decoder_output + ffn_output
        )

        return decoder_output

In [8]:
# TransformerEncoder (already in b8d8e772, moved here for order)
class TransformerEncoder(nn.Module):
    def __init__(self, vocab_size, d_model, num_heads, ffn_hidden_dim, num_layers, max_seq_len, dropout_rate=0.1):
        super().__init__()
        self.embedding = nn.Embedding(vocab_size, d_model)
        self.positional_encoding = PositionalEncoding(d_model, max_seq_len)
        self.encoder_layers = nn.ModuleList([
            TransformerEncoderLayer(d_model, num_heads, ffn_hidden_dim, dropout_rate)
            for _ in range(num_layers)
        ])

    def forward(self, src, src_mask):
        x = self.embedding(src)
        x = self.positional_encoding(x)
        for layer in self.encoder_layers:
            x = layer(x, mask=src_mask)
        return x

# TransformerDecoder (from cell p55tJHues3fz)
class TransformerDecoder(nn.Module):
    def __init__(self, vocab_size, d_model, num_heads, ffn_hidden_dim, num_layers, max_seq_len, dropout_rate=0.1):
        super().__init__()
        self.embedding = nn.Embedding(vocab_size, d_model)
        self.positional_encoding = PositionalEncoding(d_model, max_seq_len)
        self.decoder_layers = nn.ModuleList([
            TransformerDecoderLayer(d_model, num_heads, ffn_hidden_dim, dropout_rate)
            for _ in range(num_layers)
        ])

    def forward(self, tgt, enc_output, src_mask, tgt_mask):
        x = self.embedding(tgt)
        x = self.positional_encoding(x)
        for layer in self.decoder_layers:
            x = layer(x, enc_output, src_mask, tgt_mask)
        return x

# FullTransformer
class FullTransformer(nn.Module):
    def __init__(self, src_vocab_size, tgt_vocab_size, d_model, num_heads, ffn_hidden_dim, num_encoder_layers, num_decoder_layers, max_seq_len, dropout_rate=0.1):
        super().__init__()
        self.encoder = TransformerEncoder(
            src_vocab_size, d_model, num_heads, ffn_hidden_dim, num_encoder_layers, max_seq_len, dropout_rate
        )
        self.decoder = TransformerDecoder(
            tgt_vocab_size, d_model, num_heads, ffn_hidden_dim, num_decoder_layers, max_seq_len, dropout_rate
        )
        self.output_linear = nn.Linear(d_model, tgt_vocab_size)

    def forward(self, src, tgt, src_mask, tgt_mask):
        enc_output = self.encoder(src, src_mask)
        dec_output = self.decoder(tgt, enc_output, src_mask, tgt_mask)
        output = self.output_linear(dec_output)
        return output


In [9]:
# --- Updated Hyperparameters ---
d_model = 128
num_heads = 4
ffn_hidden_dim = 512
max_seq_len = 20

def create_padding_mask(seq, pad_idx):
    mask = (seq != pad_idx).unsqueeze(1).unsqueeze(2)
    return mask

def create_look_ahead_mask(seq_len, device):
    look_ahead_mask = torch.triu(torch.ones(seq_len, seq_len, device=device), diagonal=1).bool()
    return look_ahead_mask

def create_decoder_input_masks(target_seq, pad_idx):
    device = target_seq.device
    padding_mask = create_padding_mask(target_seq, pad_idx)
    seq_len = target_seq.shape[1]
    look_ahead_mask = create_look_ahead_mask(seq_len, device)
    combined_mask = padding_mask & ~look_ahead_mask
    return combined_mask.bool()

def prepare_decoder_inputs(input_text, vocab, max_seq_len):
    pad_token_id = vocab['<pad>']
    sos_token_id = vocab['<sos>']
    eos_token_id = vocab['<eos>']
    tokens = [sos_token_id]
    for word in str(input_text).lower().replace('.', '').split():
        tokens.append(vocab.get(word, vocab['<unk>']))
    tokens.append(eos_token_id)
    if len(tokens) > max_seq_len:
        tokens = tokens[:max_seq_len-1] + [eos_token_id]
    else:
        tokens.extend([pad_token_id] * (max_seq_len - len(tokens)))
    input_ids = torch.tensor(tokens, dtype=torch.long).unsqueeze(0)
    tgt_mask = create_decoder_input_masks(input_ids, pad_token_id)
    return input_ids, tgt_mask

In [10]:
import pandas as pd
import torch
import torch.nn as nn
import torch.optim as optim
import random

# --- 1. Load Data ---
file_path = '/content/drive/MyDrive/datasets/english_hindi_dataset.csv'
df = pd.read_csv(file_path).dropna()

english_sentences = df['English'].astype(str).tolist()
hindi_sentences = df['Hindi'].astype(str).tolist()

# --- 2. Build Vocabulary with lower min_freq (min_freq=5) ---
def build_vocabulary(sentences, special_tokens_dict, min_freq=5):
    word_counts = {}
    for sentence in sentences:
        for word in sentence.lower().replace('.', '').replace('?', '').split():
            word_counts[word] = word_counts.get(word, 0) + 1

    vocab = special_tokens_dict.copy()
    current_idx = len(vocab)
    for word, count in sorted(word_counts.items(), key=lambda item: item[1], reverse=True):
        if count >= min_freq and word not in vocab:
            vocab[word] = current_idx
            current_idx += 1
    return vocab, {idx: word for word, idx in vocab.items()}, len(vocab)

print("Re-building expanded vocabularies...")
english_vocab, english_idx_to_word, english_vocab_size = build_vocabulary(english_sentences, {'<pad>': 0, '<unk>': 1, '<cls>': 2, '[sep]': 3})
hindi_vocab, hindi_idx_to_word, hindi_vocab_size = build_vocabulary(hindi_sentences, {'<pad>': 0, '<unk>': 1, '<sos>': 2, '<eos>': 3})

print(f"New English Vocab Size: {english_vocab_size}, Hindi Vocab Size: {hindi_vocab_size}")

Re-building expanded vocabularies...
New English Vocab Size: 164, Hindi Vocab Size: 190


In [11]:
# --- 5. Re-instantiate Transformer with Higher Capacity ---
num_encoder_layers_full_transformer = 2
num_decoder_layers_full_transformer = 2
dropout_rate_full_transformer = 0.1

full_transformer_model = FullTransformer(
    src_vocab_size=english_vocab_size,
    tgt_vocab_size=hindi_vocab_size,
    d_model=d_model,
    num_heads=num_heads,
    ffn_hidden_dim=ffn_hidden_dim,
    num_encoder_layers=num_encoder_layers_full_transformer,
    num_decoder_layers=num_decoder_layers_full_transformer,
    max_seq_len=max_seq_len,
    dropout_rate=dropout_rate_full_transformer
).to(device)

print(f"\nFullTransformer re-created with d_model={d_model} and num_heads={num_heads} on {device}!")


FullTransformer re-created with d_model=128 and num_heads=4 on cuda!


In [12]:

# --- 6. Learning Rate Scheduler ---
class LRScheduler:
    def __init__(self, optimizer, d_model, warmup_steps=4000):
        self.optimizer = optimizer
        self.d_model = d_model
        self.warmup_steps = warmup_steps
        self.current_step = 0
        self.lr = 0

    def step(self):
        self.current_step += 1
        lr = self.d_model**(-0.5) * min(self.current_step**(-0.5), self.current_step * self.warmup_steps**(-1.5))
        for param_group in self.optimizer.param_groups:
            param_group['lr'] = lr
        self.lr = lr
        return lr


In [13]:
# --- 7. Real Data Batch Generator ---
def generate_batch(eng_sentences, hin_sentences, eng_vocab, hin_vocab, max_seq_len, batch_size):
    src_input_ids_batch = []
    src_attention_mask_batch = []
    tgt_input_ids_batch = []
    tgt_mask_batch = []

    # Pick random indices to form a batch of parallel sentences
    batch_indices = random.sample(range(len(eng_sentences)), batch_size)

    for i in batch_indices:
        # Source (English)
        src_input_ids, src_mask, _ = prepare_transformer_inputs(
            eng_sentences[i], eng_vocab, max_seq_len
        )
        src_input_ids_batch.append(src_input_ids)
        src_attention_mask_batch.append(src_mask)

        # Target (Hindi)
        tgt_input_ids, tgt_mask = prepare_decoder_inputs(
            hin_sentences[i], hin_vocab, max_seq_len
        )
        tgt_input_ids_batch.append(tgt_input_ids)
        tgt_mask_batch.append(tgt_mask)

    # Fix: Return the tuple of tensors. Do not call .to(device) on the tuple itself.
    return (
        torch.cat(src_input_ids_batch, dim=0),
        torch.cat(src_attention_mask_batch, dim=0),
        torch.cat(tgt_input_ids_batch, dim=0),
        torch.cat(tgt_mask_batch, dim=0)
    )

In [14]:
!pip install torchmetrics
import torchmetrics
import pandas as pd
from torch.utils.data import Dataset, DataLoader, random_split

# --- 0. Ensure Data is Loaded ---
# If for some reason the data cell wasn't run, we handle it here
if 'english_sentences' not in globals():
    print("Loading dataset...")
    file_path = '/content/drive/MyDrive/datasets/english_hindi_dataset.csv'
    df = pd.read_csv(file_path).dropna()
    english_sentences = df['English'].astype(str).tolist()
    hindi_sentences = df['Hindi'].astype(str).tolist()

# --- 1. Dataset Preparation ---
class TranslationDataset(Dataset):
    def __init__(self, src_sentences, tgt_sentences, src_vocab, tgt_vocab, max_seq_len):
        self.src_sentences = src_sentences
        self.tgt_sentences = tgt_sentences
        self.src_vocab = src_vocab
        self.tgt_vocab = tgt_vocab
        self.max_seq_len = max_seq_len

    def __len__(self):
        return len(self.src_sentences)

    def __getitem__(self, idx):
        src_ids, src_mask, _ = prepare_transformer_inputs(self.src_sentences[idx], self.src_vocab, self.max_seq_len)
        tgt_ids, tgt_mask = prepare_decoder_inputs(self.tgt_sentences[idx], self.tgt_vocab, self.max_seq_len)
        # Ensure indices are within bounds
        src_ids = torch.clamp(src_ids, 0, english_vocab_size - 1)
        tgt_ids = torch.clamp(tgt_ids, 0, hindi_vocab_size - 1)
        return src_ids.squeeze(0), src_mask.squeeze(0), tgt_ids.squeeze(0), tgt_mask.squeeze(0)

# Prepare data loaders
max_seq_len = 20
full_dataset = TranslationDataset(english_sentences, hindi_sentences, english_vocab, hindi_vocab, max_seq_len)
train_size = int(0.8 * len(full_dataset))
val_size = len(full_dataset) - train_size
train_subset, val_subset = random_split(full_dataset, [train_size, val_size])

train_loader = DataLoader(train_subset, batch_size=32, shuffle=True)

# --- 2. Model Initialization ---
training_device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')

model = FullTransformer(
    src_vocab_size=english_vocab_size,
    tgt_vocab_size=hindi_vocab_size,
    d_model=256,
    num_heads=8,
    ffn_hidden_dim=512,
    num_encoder_layers=4,
    num_decoder_layers=4,
    max_seq_len=max_seq_len,
    dropout_rate=0.1
).to(training_device)

optimizer = torch.optim.Adam(model.parameters(), lr=0.0001)
criterion = nn.CrossEntropyLoss(ignore_index=0)
accuracy_metric = torchmetrics.Accuracy(task="multiclass", num_classes=hindi_vocab_size, ignore_index=0).to(training_device)

# --- 3. Training Loop ---
print(f"Training with Vocab Sizes -> English: {english_vocab_size}, Hindi: {hindi_vocab_size}")
for epoch in range(10):
    model.train()
    total_loss = 0
    accuracy_metric.reset()

    for src, src_m, tgt, tgt_m in train_loader:
        src, src_m, tgt, tgt_m = src.to(training_device), src_m.to(training_device), tgt.to(training_device), tgt_m.to(training_device)

        dec_input = tgt[:, :-1]
        targets = tgt[:, 1:]
        dec_mask = tgt_m[:, :, :-1, :-1]

        optimizer.zero_grad()
        output = model(src, dec_input, src_m, dec_mask)

        loss = criterion(output.reshape(-1, hindi_vocab_size), targets.reshape(-1))
        loss.backward()
        optimizer.step()

        total_loss += loss.item()
        accuracy_metric.update(output.reshape(-1, hindi_vocab_size), targets.reshape(-1))

    print(f"Epoch {epoch+1}: Loss = {total_loss/len(train_loader):.4f}, Accuracy = {accuracy_metric.compute():.4f}")

Training with Vocab Sizes -> English: 164, Hindi: 190
Epoch 1: Loss = 3.5920, Accuracy = 0.2960
Epoch 2: Loss = 1.9374, Accuracy = 0.6341
Epoch 3: Loss = 1.0448, Accuracy = 0.8376
Epoch 4: Loss = 0.5985, Accuracy = 0.8926
Epoch 5: Loss = 0.4146, Accuracy = 0.9152
Epoch 6: Loss = 0.3146, Accuracy = 0.9358
Epoch 7: Loss = 0.2456, Accuracy = 0.9490
Epoch 8: Loss = 0.2001, Accuracy = 0.9543
Epoch 9: Loss = 0.1681, Accuracy = 0.9567
Epoch 10: Loss = 0.1452, Accuracy = 0.9614


In [21]:
def translate_sentence(sentence, model, src_vocab, tgt_vocab, max_seq_len, device):
    model.eval()

    # 1. Prepare source input
    src_ids, src_mask, _ = prepare_transformer_inputs(sentence, src_vocab, max_seq_len)
    src_ids, src_mask = src_ids.to(device), src_mask.to(device)

    # 2. Initialize decoder input with <sos> token
    # We start with a single token and append predictions one by one
    decoder_input = torch.tensor([[tgt_vocab['<sos>']]], dtype=torch.long).to(device)

    for _ in range(max_seq_len):
        # Create mask for current decoder sequence
        tgt_mask = create_decoder_input_masks(decoder_input, tgt_vocab['<pad>'])

        with torch.no_grad():
            # Forward pass through the full model
            output = model(src_ids, decoder_input, src_mask, tgt_mask)

        # 3. Get the prediction for the last time step
        next_token_logits = output[:, -1, :]
        next_token = torch.argmax(next_token_logits, dim=-1).unsqueeze(0)

        # Append predicted token to decoder input
        decoder_input = torch.cat([decoder_input, next_token], dim=1)

        # 4. Stop if model predicts End of Sentence
        if next_token.item() == tgt_vocab['<eos>']:
            break

    # 5. Convert token IDs back to words
    translated_tokens = [hindi_idx_to_word[idx.item()] for idx in decoder_input[0]]

    # Clean up special tokens for the final output
    final_sentence = " ".join([t for t in translated_tokens if t not in ['<sos>', '<eos>', '<pad>']])
    return final_sentence

# --- Test the translation ---
test_sentence = "You are  "
# Ensure we use the 'model' variable initialized in the training cell
prediction = translate_sentence(test_sentence, model, english_vocab, hindi_vocab, max_seq_len, training_device)

print(f"English: {test_sentence}")
print(f"Hindi Prediction: {prediction}")

English: You are  
Hindi Prediction: तुम तुम हो।
